# Developing ML Classification Framework: Less Imbalanced Biotic-Abiotic Sample Set

##### Introduction:
In this Jupyter notebook, four ML classifiers are trained and evaluated on pyrolysis-Gas Chromatography-Mass Spectrometry (py-GC-MS) mass-spectral datasets. In order to evaluate each classifier's performance under a less severe class imbalance, this work uses a subset of 32 biotic and 14 abiotic py-GC-MS samples, where information on the entire sample dataset is detailed in the following [documentation](../../docs/total-sample-breakdown.md). Using a more manageable dataset, this notebook aims to determine whether geochemical patterns from the sample set can be used to distinguish between biologically-generated material and non-biological analogues before extending this task to the complete imbalanced sample set in the [next notebook](04_imbalanced_ml_classification.ipynb). The usage of ML methods for such classification tasks is vital for astrobiology research, especially as we enter a new age of space missions focused on life detection beyond Earth.

##### Imports:

In [1]:
# mlbiosig imports:
from mlbiosig import samples_labels, build_features, engineer_features

# Model imports:
from sklearn.linear_model import LogisticRegression  # LR
from sklearn.ensemble import RandomForestClassifier  # RF
from sklearn.svm import SVC  # SVC
from xgboost import XGBClassifier  # XGB

# ML imports:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    RocCurveDisplay,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Misc. imports:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint
import warnings  # hide warnings

##### Generating py-MS Feature Matrix

The py-MS feature matrix for 32 biotic and 14 abiotic samples will be generated, and involves manually choosing the biotic samples that best represent the chemical diversity of the sample set.

In [2]:
# Obtain biotic & abiotic py-GC-MS mass spectra CSV filepaths:
filepaths, labels = samples_labels()  # filepaths and label ('biotic'/'abiotic')
print(
    f"Total Samples: {len(filepaths)} | Total Biotic: {labels.count('biotic')}| Total Abiotic: {labels.count('abiotic')}"
)

Total Samples: 154 | Total Biotic: 140| Total Abiotic: 14


In [5]:
# Chosen 32 biotic sample filenames:
microbials = [
    "041121_SP_0",
    "091121_SP_21H",
    "101221_SP_48H_1",
    "200122_SP_NACL_24H",  # spirulina
    "241121_CH",
    "081221_CH_24H",
    "CHLORBACT_MGCL2_110825",  # chlorella
    "SHEW_1",
    "SHEW_2",
    "SHEW_3",
    "SHEW_4",
    "SHEW_5",
]  # shewanella

biopolymers = [
    "MEMBRANE_LIPID",
    "MEMBRANE_LIPID_MGCL2_150",  # membrane lipids
    "CELLULOSE_20231016",  # cellulose
    "LIGNIN_20231018",
]  # biotic lignin

biochar = [
    "BARLEYSTRAW_150_231106",  # barley straw
    "CHESTNUTWOOD_300_231107",  # chestnut wood
    "RICE_HUSK_200_20231122",  # rice husk
    "PINE_BARK_200_20231124",  # pine bark
    "EUCALYPTUSBARK_300_231109",
]  # eucalyptus

coals = ["DM_031125", "HIGHVOLDM1"]

kerogens = [
    "080206GRSSPLT",  # green river shale
    "CBN1_4229.4FT_650_WHOLE",
    "CD3_650_WHOLE",
    "LF1_23-4M_650_WHOLE",
    "RMP1_665-4M_650_WHOLE",
]  # kerogens

waxes_resins = [
    "AMBER_140726_3",
    "FRANK_140726_2",  # resins
    "BEESWAXNAT181018_2",
    "CARNAUBA181018_4",
]  # waxes

biotic_names = np.concatenate(
    (microbials, biopolymers, biochar, coals, kerogens, waxes_resins), axis=None
)

# Re-packaging biotic & abiotic filepaths + corresponding label:
balanced_filepaths = []
balanced_labels = []

for filepath, label in zip(filepaths, labels):
    filename = filepath.split("/")[-1].replace(".CSV", "")
    if label == "abiotic" or filename in biotic_names:  # same number of abiotic
        balanced_filepaths.append(filepath)
        balanced_labels.append(label)

print(
    f"Subset Samples: {len(balanced_filepaths)} | Subset Biotic: {balanced_labels.count('biotic')} | Total Abiotic: {balanced_labels.count('abiotic')}"
)

Subset Samples: 46 | Subset Biotic: 32 | Total Abiotic: 14


In [6]:
# Build feature matrix:
feature_matrix, flags = build_features(balanced_filepaths, balanced_labels)

feature_matrix.head(5)  # first 5 samples

Preprocessing py-GC-MS samples:   0%|          | 0/46 [00:00<?, ?sample/s]

m_z,50.1,50.3,50.8,51.0,51.1,51.3,52.0,52.1,52.2,52.3,...,14.8,19.4,14.1,16.3,17.3,27.1,28.2,29.3,32.2,40.0
041121_SP_0,0.196884,0.000166,0.000001,0.000445,0.350330,0.000039,0.000741,0.155981,0.155879,0.000102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
080206GRSSPLT,0.015949,0.000000,0.000000,0.000000,0.080974,0.000000,0.000000,0.040579,0.000132,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
081221_CH_24H,0.167491,0.000020,0.000000,0.000136,0.307936,0.000004,0.000050,0.228654,0.040986,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
091121_SP_21H,0.175225,0.000022,0.000000,0.000034,0.282388,0.000000,0.000224,0.294299,0.000000,0.000019,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
101221_SP_48H_1,0.163892,0.000009,0.000000,0.000029,0.306565,0.000009,0.000037,0.276287,0.000000,0.000006,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


##### Performing Feature Engineering

As the dataset contains a limited number of samples in total, classifier performance will be evaluated using stratified cross-validation (CV) with five folds during hyperparameter tuning.

In [21]:
# Separate features & target, split test/train, engineer features ready for ML:
X_train, X_test, y_train, y_test, le, selector = engineer_features(feature_matrix)
print(f"Training set: {len(X_train)} | Testing set: {len(X_test)}")

Training set: 36 | Testing set: 10


In [8]:
# Stratified CV folds due to (a) imbalanced and (b) a small dataset:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

##### ML Biotic-Abiotic Classification: Overview

To determine which model archicture best separates biotic and abiotic samples, four ML classifiers will be compared:
- Linear baseline models: Logistic Regression (**LR**), Support Vector Classifier (**SVC**)
- Tree-based bagging ensemble: Random Forest (**RF**) Classifier
- Tree-based gradient boosting ensemble: **XGBoost**

While RF and XGBoost are both tree-based classifiers, they will be compared on the basis of their ensemble method. RF constructs independent decision trees using bootstrap samples of the training data to ultimately aggregate their predictions, while XGBoost sequentially trains multiple decision trees which iteratively correct the errors of previous trees. The motivation behind using RF is driven by its strong performance for distinguishing between biotic and abiotic py-GC-MS samples (Cleaves 2023, Slaughter 2026). SVC is also included due to their effectiveness in high-dimensional spaces, especially when the number of features ($225$ $m/z$ values) exceeds the number of samples ($46$ in this notebook).

##### Handling Biotic-Abiotic Imbalance

For SVC, Logistic Regression, and RF, class imbalance is addressed by setting the `class_weight` hyperparameter to `"balanced"`, in order to penalise minority errors more by assigning larger weights. For XGBoost, class imbalance is handled using `scale_pos_weight` which controls the balance of positive (biotic) and negative (abiotic) weights within the loss function. This is calculated using the ratio of abiotic-to-biotic samples.

In [9]:
# Calculating class imbalance ratio for XGBoost:
abiotic_samples = (y_train == 0).sum()  # abiotic encoded as 0
biotic_samples = (y_train == 1).sum()  # biotic encoded as 1

imbalance_ratio = abiotic_samples / biotic_samples
print(f"Class imbalance ratio (scale_pos_weight): {imbalance_ratio:.3f}")

Class imbalance ratio (scale_pos_weight): 0.440


##### Feature Scaling

Scaling is specifically performed due to the inclusion of `SVC()` and `LogisticRegression()` as they are not robust against outliers, where an extensive discussion in the previous [notebook](02_feature_engineering.ipynb) further explains and demonstrates why outliers were not capped or removed. As the sample dataset will be split into CV folds, scaling is not performed during feature engineering to prevent data leakage. Moreover, the selected scaler is chosen based on its compatibility with a given classifier. While tree-based models, such as RF and XGBoost, are insensitive to scaling, the same pipeline is applied across each model to maintain consistency.


| Model | Scaler | Description |
| ------- | -------------- | ------------|
| `LogisticRegression()` | `StandardScaler()` | Z-score normalisation is most effective for data assumed to be normally distributed. |
| `SVC()` | `StandardScaler` | Normalisation is best for distance-based algorithms since feature magnitude directly influences the model. `MinMaxScaler` was initially chosen but ultimately replaced due to repeated convergence warnings.  |
|`RandomForestClassifier()` | `RobustScaler()` | Tree-based models are insensitive to feature scaling, hence robust scaling is employed for consistency while reducing the effect of extreme outliers. |
|`XGBoost()` | `RobustScaler()` |Tree-based models are insensitive to feature scaling, hence robust scaling is employed for consistency while reducing the effect of extreme outliers. |


##### Building ML Pipelines

A separate `Pipeline` is created for each of the four classifiers, where preprocessing and modeling training are combined. This also ensures that scaling and class imbalance are simultaenously handled within each CV fold, in order to prevent data leakage during hyperparameter tuning.

In [10]:
# Logistic Regression:
lr_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(class_weight="balanced", random_state=42, max_iter=500),
        ),
    ]
)

# Support Vector Classifier (SVC):
svc_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "classifier",
            SVC(
                class_weight="balanced", random_state=42, max_iter=500, probability=True
            ),
        ),
    ]
)

# Random Forest:
rf_pipeline = Pipeline(
    [
        ("scaler", RobustScaler()),
        (
            "classifier",
            RandomForestClassifier(class_weight="balanced", random_state=42),
        ),
    ]
)

# XGBoost:
xgb_pipeline = Pipeline(
    [
        ("scaler", RobustScaler()),
        (
            "classifier",
            XGBClassifier(
                scale_pos_weight=imbalance_ratio,
                random_state=42,
                eval_metric="logloss",  # binary cross entropy for comparisons
            ),
        ),
    ]
)

##### Hyperparameter Tuning

Before comparing the performance of the four models, the hyperparameter combinations of each classifier is optimised to best maximise its performance. A coarse-to-fine hyperparameter tuning strategy is adopted as follows:
* First exploring a broad hyperparameter space that results in a generally good performance via `RandomizedSearchCSV`. 
* Identifying the best-performing region during the random search and then fine-tuning to tighter ranges using `GridSearchCV` over a narrower search space.
* The hyperparameter values that result in the highest CV F1 score will be selected for each classifier.

F1 score is the chosen comparison metric for determining and selecting the best hyperparameter combinations, where F1 $\in [0, 1]$ and is calculated via

$$ 
\text{F}1 = \frac{2 \cdot (\text{Precision} \times \text{Recall})}{\text{Precision} + \text{Recall}}
$$  

where precision represents how often the model correctly predicts a positive result i.e., biotic and recall captures how many true biotic samples were found (Christen 2023). 

For `LogisticRegression()`:

<div align="center">

| Hyperparameter | Randomised Search Range | Description |
| -------------- | ------------------------| ------------| 
|`C` | $C \in [0.001, 0.01, 0.1, 1, 10, 100]$ | Regularisation controls the trade-off between model complexity and overfitting, where smaller values impose stronger regularisation. |
|`solver` | `lbfgs`, `liblinear` | Algorithm used to fit data i.e, `lbfgs` as the default or `liblinear` for high-dimensional smaller datasets.|

</div>

`C` is a critical hyperparameter in this work, as regularisation will handle the high multicollinearity of features (shown in the [previous notebook](02_feature_engineering.ipynb)) as they can cause instability in coefficient estimation (Vatcheva 2016).


In [11]:
# Hide FutureWarning for better readability:
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.svm")

In [12]:
# Coarse Tuning via RandomSearch:
lr_random_param = {
    "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "classifier__solver": ["lbfgs", "liblinear"],
}

# Perform search and fit on training data:
lr_random_search = RandomizedSearchCV(
    lr_pipeline,
    lr_random_param,
    cv=cv,
    scoring="f1",  # F1 scoring
    random_state=42,  # for reproducability
    n_jobs=-1,  # use all CPUs
)
lr_random_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"LR Random Search Best Params: {lr_random_search.best_params_}")  # best params
print(f"LR Random Search Best CV F1: {lr_random_search.best_score_:.3f}")  # best F1

best_C_lr = lr_random_search.best_params_["classifier__C"]
best_solver_lr = lr_random_search.best_params_["classifier__solver"]

LR Random Search Best Params: {'classifier__solver': 'lbfgs', 'classifier__C': 10}
LR Random Search Best CV F1: 0.923


In [13]:
# Fine Tuning via GridSearch: Dependent on random search space
lr_grid_param = {
    "classifier__C": [
        best_C_lr * 0.1,
        best_C_lr * 0.5,
        best_C_lr,  # based on random search
        best_C_lr * 2,
        best_C_lr * 10,
    ],
    "classifier__solver": [best_solver_lr],
}

# Perform search and fit on training data:
lr_grid_search = GridSearchCV(
    lr_pipeline, lr_grid_param, cv=cv, scoring="f1", n_jobs=-1  # use all CPUs
)
lr_grid_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"LR Fine-Tuned Grid Search Best Params: {lr_grid_search.best_params_}")
print(f"LR Fine-Tuned Search Best CV F1: {lr_grid_search.best_score_:.3f}")  # best F1

LR Fine-Tuned Grid Search Best Params: {'classifier__C': 1.0, 'classifier__solver': 'lbfgs'}
LR Fine-Tuned Search Best CV F1: 0.923


For `SVC()`:

<div align="center">

| Hyperparameter | Randomised Search Range | Description |
| -------------- | ------------------------| ------------|
|`C` | $C \in [0.001, 0.01, 0.1, 1, 10, 100]$ | Regularisation controls the trade-off between model complexity and overfitting. |
|`kernel` | `linear`, `rbf` | Kernel function used to map input data into a feature space i.e., `linear`, `rbf`.|
| `gamma`| `scale`, `auto` | Considered for non-linear kernels i.e., `kernel = rbf`, controls the influence of training data on the decision boundary.|

</div>

In [14]:
# Coarse Tuning via RandomSearch:
svc_random_param = {
    "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "classifier__kernel": ["linear", "rbf"],
    "classifier__gamma": ["scale", "auto"],
}

# Perform search and fit on training data:
svc_random_search = RandomizedSearchCV(
    svc_pipeline,
    svc_random_param,
    cv=cv,
    scoring="f1",
    random_state=42,  # for reproducability
    n_jobs=-1,  # use all CPUs
)
svc_random_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"SVC Random Search Best Params: {svc_random_search.best_params_}")  # best params
print(f"SVC Random Search Best CV F1: {svc_random_search.best_score_:.3f}")  # best F1

best_C_svc = svc_random_search.best_params_["classifier__C"]
best_kernel_svc = svc_random_search.best_params_["classifier__kernel"]
best_gamma_svc = svc_random_search.best_params_["classifier__gamma"]

SVC Random Search Best Params: {'classifier__kernel': 'linear', 'classifier__gamma': 'scale', 'classifier__C': 10}
SVC Random Search Best CV F1: 0.923


In [15]:
# Fine Tuning via GridSearch: Dependent on random search space
svc_grid_param = {
    "classifier__C": [
        best_C_svc * 0.1,
        best_C_svc * 0.5,
        best_C_svc,  # based on random search
        best_C_svc * 2,
        best_C_svc * 10,
    ],
    "classifier__kernel": [best_kernel_svc],
    "classifier__gamma": [best_gamma_svc],
}

# Perform search and fit on training data:
svc_grid_search = GridSearchCV(
    svc_pipeline, svc_grid_param, cv=cv, scoring="f1", n_jobs=-1  # use all CPUs
)
svc_grid_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"SVC Fine-Tuned Grid Search Best Params: {svc_grid_search.best_params_}")
print(f"SVC Fine-Tuned Search Best CV F1: {svc_grid_search.best_score_:.3f}")  # best F1

SVC Fine-Tuned Grid Search Best Params: {'classifier__C': 1.0, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear'}
SVC Fine-Tuned Search Best CV F1: 0.923


For `RandomForestClassifier()`:

<div align="center">

| Hyperparameter | Randomised Search Range | Description |
| -------------- | ------------------------| ------------|
|`n_estimators` | `randint(50, 500)` | Number of decision trees in forest.|
| `max_depth`| `None`, $5, 10, 15, 20, 30, 50$ | Maximum depth of each decision tree. |
|`min_samples_split` | `randint(5, 20)` | Minimum number of samples required to split node.|

</div>

In [16]:
# Coarse Tuning via RandomSearch:
# many of these will be sampled randomly
rf_random_param = {
    "classifier__n_estimators": randint(50, 500),
    "classifier__max_depth": [None, 5, 10, 15, 20, 30, 50],
    "classifier__min_samples_split": randint(5, 20),
}

# Perform search and fit on training data:
rf_random_search = RandomizedSearchCV(
    rf_pipeline,
    rf_random_param,
    cv=cv,
    scoring="f1",
    random_state=42,  # for reproducability
    n_jobs=-1,  # use all CPUs
)
rf_random_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"RF Random Search Best Params: {rf_random_search.best_params_}")  # best params
print(f"RF Random Search Best CV F1: {rf_random_search.best_score_:.3f}")  # best F1

best_n_rf = rf_random_search.best_params_["classifier__n_estimators"]
best_depth_rf = rf_random_search.best_params_["classifier__max_depth"]
best_minsamples_rf = rf_random_search.best_params_["classifier__min_samples_split"]

RF Random Search Best Params: {'classifier__max_depth': 50, 'classifier__min_samples_split': 8, 'classifier__n_estimators': 398}
RF Random Search Best CV F1: 0.893


In [17]:
# Fine Tuning via GridSearch: Dependent on random search space
rf_grid_param = {
    "classifier__n_estimators": [best_n_rf - 100, best_n_rf, best_n_rf + 100],
    "classifier__max_depth": [
        best_depth_rf - 5,
        best_depth_rf - 4,
        best_depth_rf - 3,
        best_depth_rf,
    ],
    "classifier__min_samples_split": [
        max(2, best_minsamples_rf - 2),
        best_minsamples_rf,
        best_minsamples_rf + 2,
    ],
}

# Perform search and fit on training data:
rf_grid_search = GridSearchCV(
    rf_pipeline, rf_grid_param, cv=cv, scoring="f1", n_jobs=-1  # use all CPUs
)
rf_grid_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"RF Fine-Tuned Grid Search Best Params: {rf_grid_search.best_params_}")
print(f"RF Fine-Tuned Search Best CV F1: {rf_grid_search.best_score_:.3f}")  # best F1

RF Fine-Tuned Grid Search Best Params: {'classifier__max_depth': 45, 'classifier__min_samples_split': 6, 'classifier__n_estimators': 298}
RF Fine-Tuned Search Best CV F1: 0.893


For `XGBClassifier`:

<div align="center">

| Hyperparameter |  Randomised Search Range | Description | 
| -------------- | ------------------------ | ------------|
|`n_estimators` | `randint(50, 500)` | Number of trees (or boosting rounds).|
|`learning_rate` | $0.001, 0.01, 0.1, 0.3, 0.5$ |  Determines the step size during boosting, where the default value is $0.3$.|
| `max_depth`| `None`, $5, 10, 15, 20, 30, 50$ | Maximum depth of each decision tree. |

</div>

In [ ]:
# Coarse Tuning via RandomSearch:
xgb_random_param = {
    "classifier__n_estimators": randint(50, 500),
    "classifier__learning_rate": [0.001, 0.01, 0.1, 0.3, 0.5],
    "classifier__max_depth": [None, 5, 10, 15, 20, 30, 50],
}

# Perform search and fit on training data:
xgb_random_search = RandomizedSearchCV(
    xgb_pipeline,
    xgb_random_param,
    cv=cv,
    scoring="f1",
    random_state=42,  # for reproducability
    n_jobs=-1,  # use all CPUs
)
xgb_random_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"XGB Random Search Best Params: {xgb_random_search.best_params_}")  # best params
print(f"XGB Random Search Best CV F1: {xgb_random_search.best_score_:.3f}")  # best F1

best_lrate_xgb = xgb_random_search.best_params_["classifier__learning_rate"]
best_n_xgb = xgb_random_search.best_params_["classifier__n_estimators"]
best_depth_xgb = xgb_random_search.best_params_["classifier__max_depth"]

XGB Random Search Best Params: {'classifier__learning_rate': 0.01, 'classifier__max_depth': 15, 'classifier__n_estimators': 463}
XGB Random Search Best CV F1: 0.788


In [19]:
# Fine Tuning via GridSearch: Dependent on random search space
xgb_grid_param = {
    "classifier__learning_rate": [
        best_lrate_xgb * 0.5,
        best_lrate_xgb,
        best_lrate_xgb * 0.8,
    ],
    "classifier__n_estimators": [best_n_xgb - 100, best_n_xgb, best_n_xgb + 100],
    "classifier__max_depth": [
        best_depth_xgb - 5,
        best_depth_xgb - 4,
        best_depth_xgb - 3,
        best_depth_xgb,
    ],
}

# Perform search and fit on training data:
xgb_grid_search = GridSearchCV(
    xgb_pipeline, xgb_grid_param, cv=cv, scoring="f1", n_jobs=-1  # use all CPUs
)
xgb_grid_search.fit(X_train, y_train)

# Best parameter combination & F1 score:
print(f"XGB Fine-Tuned Grid Search Best Params: {xgb_grid_search.best_params_}")
print(f"XGB Fine-Tuned Search Best CV F1: {xgb_grid_search.best_score_:.3f}")  # best F1

XGB Fine-Tuned Grid Search Best Params: {'classifier__learning_rate': 0.005, 'classifier__max_depth': 10, 'classifier__n_estimators': 563}
XGB Fine-Tuned Search Best CV F1: 0.788


##### Comparing ML Models with Best Hyperparameters

Having tuned the model hyperparameters, each classifier can be refitted with the optimal hyperparameter values on the entire training dataset. The final trained models will then be evaluated once on the test set, in order to assess performance with no bias. Model performance will continue to be examined using the F1 score, and the Receiver Operating Characteristic Area Under the Curve (ROC AUC) to determine whether each model can correctly distinguish between the biotic and abiotic class (score closer to $1$) better than random guessing ($\sim 0.5$). Overall accuracy is not evaluated, given the class imbalance of biotics heavily outnumbering abiotics.

In [20]:
# Dictionary gathering best hyperparameters across all four models:
models = {
    "Logistic Regression": lr_grid_search.best_estimator_,
    "SVC": svc_grid_search.best_estimator_,
    "Random Forest": rf_grid_search.best_estimator_,
    "XGBoost": xgb_grid_search.best_estimator_,
}

results = {}  # dict for results
for c_name, model in models.items():
    y_pred = model.predict(X_test)  # predictions on testing set
    y_prob = model.predict_proba(X_test)[:, 1]  # probability of predictions
    f1 = f1_score(y_test, y_pred)  # F1 score
    auc = roc_auc_score(y_test, y_prob)  # AUC ROC score
    # Across 5 CV folds, evaluate score:
    cv_f1 = {
        "Logistic Regression": lr_grid_search.best_score_,
        "SVC": svc_grid_search.best_score_,
        "Random Forest": rf_grid_search.best_score_,
        "XGBoost": xgb_grid_search.best_score_,
    }[c_name]
    results[c_name] = {"cv_f1": cv_f1, "f1": f1, "auc": auc}
    # Output results:
    print(f"\n{c_name} Results:")
    print(f"CV F1: {cv_f1:.3f} | F1: {f1:.3f} | AUC ROC: {auc:.3f}")


Logistic Regression Results:
CV F1: 0.923 | F1: 0.769 | AUC ROC: 0.857

SVC Results:
CV F1: 0.923 | F1: 0.800 | AUC ROC: 0.762

Random Forest Results:
CV F1: 0.893 | F1: 1.000 | AUC ROC: 1.000

XGBoost Results:
CV F1: 0.788 | F1: 0.769 | AUC ROC: 0.762


##### Conclusions:

For this preliminary subset of $46$ py-GC-MS samples, the class imbalance between abiotic and biotic samples is minimised from $10:1$ to $\sim 2.3:1$. The performance of the four classifiers (LR, SVC, RF, XGBoost) is incredibly varied across both cross-validation and the testing set. The CV F1 scores are broadly consistent across LR, SVC, and RF, while XGBoost performs considerably lower at $\sim 0.79$. This may be due to the linear classifiers LR and SVC (`kernel=linear`), respectively, exploiting the apparent linear separation of different sample-type clusters within PCA and t-SNE explored in a [previous notebook](02_feature_engineering.ipynb). 

However, test set evaluation reveals a perfect F1 and AUC-ROC score of $1$ achieved by RF, which should be interpreted with caution due to the limited size of the test set at only $10$ samples. Hence, for this subset, CV F1 score provides a more trustworthy estimate of model performance and motivates extending model evaluation to the entire biotic-abiotic sample set in the [next notebook](04_imbalanced_ml_classification.ipynb).

##### Bibliography:

Cleaves, H. J., Hystad, G., Prabhu, A., et al. (2023). *A robust, agnostic molecular biosignature based on machine learning*. Proc. Natl. Acad. Sci. U.S.A. 120 (41). https://doi.org/10.1073/pnas.2307149120

Slaughter, E., Hystad, G., Cody, G., et al. (2026). *Detecting signs of life in biotic–abiotic mixtures using pyrolysis–gas chromatography–mass spectrometry and machine learning*. Front. Astron. Space Sci. 13:1695325. https://doi.org/10.3389/fspas.2026.1695325

Christen, P., Hand, D. J., Kirielle, N. (2023). *A Review of the F-Measure: Its History, Properties, Criticism, and Alternatives*. ACM Comput. Surv. 56, 3, Article 73. https://doi.org/10.1145/3606367

Vatcheva, K.P., Lee, M., McCormick, J.B., Rahbar, M.H. (2016) *Multicollinearity in Regression Analyses Conducted in Epidemiologic Studies*. Epidemiol 6:227. https://doi.org/10.4172/2161-1165.1000227